In [1]:
import pygame
import csv
import time
import random
import os
import json

# --- CONFIGURATION ---
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 600
CURSOR_RADIUS = 20
TARGET_RADIUS = 30
FPS = 60
FILE_NAME = "experiment_data.csv"
RECORDING_DIR = "recordings"
NUM_ITERATIONS = 20

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED   = (255, 0, 0)
BLUE  = (0, 0, 255)
GREEN = (0, 255, 0)
GRAY  = (128, 128, 128)

# Game Modes
INDIVIDUAL = "individual"
COOPERATIVE = "cooperative"
PLAYBACK = "playback"
AI = "ai"

# AI Control modes (which axis the AI controls)
AI_CONTROLS_VERTICAL = 1    # AI controls up/down
AI_CONTROLS_HORIZONTAL = 0  # AI controls left/right

# Preset target locations for 20 iterations
PRESET_TARGETS = [
    [150, 100], [650, 100], [400, 150], [200, 250], [600, 250],
    [100, 350], [700, 350], [750, 80], [550, 420], [300, 450],
    [450, 80], [550, 100], [200, 200], [650, 200], [120, 480],
    [150, 400], [700, 450], [250, 500], [600, 480], [350, 520]
]

class ExperimentGame:
    def __init__(self, mode=INDIVIDUAL, recording_file=None, iteration=1, ai_axis=None, target_pos=None, playback_axis=None):
        pygame.init()
        self.screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
        pygame.display.set_caption("CogSci Joint Action Task")
        self.clock = pygame.time.Clock()
        self.running = True
        self.mode = mode
        self.iteration = iteration
        self.target_hit = False
        
        # Positions
        self.cursor_pos = [SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2]
        if target_pos is not None:
            self.target_pos = target_pos
        else:
            self.target_pos = PRESET_TARGETS[iteration - 1]
        
        # Velocity summing variables
        self.human_v = [0, 0]
        self.partner_v = [0, 0] # Can be Human 2, AI, or Playback
        
        # Data Logging Setup
        self.data_log = []
        self.start_time = time.time()
        self.recording_file = recording_file
        
        # Playback Setup
        self.playback_data = {}
        self.playback_index = 0
        self.playback_axis = playback_axis  # Pre-determined axis
        self.recording_target_pos = None  # Target position from recording
        if mode == PLAYBACK and recording_file:
            self.load_recording(recording_file)
        
        # AI Setup
        self.ai_control_axis = ai_axis if ai_axis is not None else random.randint(0, 1)

    def load_recording(self, filename):
        """Load pre-recorded movements from a JSON file"""
        try:
            with open(filename, 'r') as f:
                recording_data = json.load(f)
            
            if 'iterations' in recording_data:
                iterations = recording_data.get('iterations', [])
                # Convert frames format to velocity arrays for playback
                converted_iterations = []
                for iter_data in iterations:
                    frames = iter_data.get('frames', [])
                    if frames:  # New format with frames
                        partner_horizontal = [f['partner_vx'] for f in frames]
                        partner_vertical = [f['partner_vy'] for f in frames]
                    else:  # Old format with velocity arrays
                        partner_horizontal = iter_data.get('partner_horizontal', [])
                        partner_vertical = iter_data.get('partner_vertical', [])
                    converted_iterations.append({
                        'iteration': iter_data.get('iteration', 0),
                        'target_pos': iter_data['target_pos'],
                        'partner_horizontal': partner_horizontal,
                        'partner_vertical': partner_vertical,
                        'horizontal': partner_horizontal,
                        'vertical': partner_vertical
                    })
                self.playback_data = {
                    'type': 'session',
                    'iterations': converted_iterations
                }
                if 1 <= self.iteration <= len(converted_iterations):
                    self.recording_target_pos = converted_iterations[self.iteration - 1].get('target_pos', self.target_pos)
                    self.target_pos = self.recording_target_pos
                else:
                    self.recording_target_pos = self.target_pos
                print(f"Loaded session recording with {len(converted_iterations)} iterations")
            else:
                self.recording_target_pos = recording_data.get('target_pos', self.target_pos)
                human_horizontal = recording_data.get('human_horizontal', [])
                human_vertical = recording_data.get('human_vertical', [])
                partner_horizontal = recording_data.get('partner_horizontal', recording_data.get('horizontal', []))
                partner_vertical = recording_data.get('partner_vertical', recording_data.get('vertical', []))
                self.playback_data = {
                    'type': 'single',
                    'human_horizontal': human_horizontal,
                    'human_vertical': human_vertical,
                    'partner_horizontal': partner_horizontal,
                    'partner_vertical': partner_vertical,
                    'horizontal': partner_horizontal,
                    'vertical': partner_vertical,
                    'target_pos': self.recording_target_pos
                }
                print(f"Loaded recording with {len(partner_horizontal)} partner horizontal frames and {len(partner_vertical)} partner vertical frames")
        except FileNotFoundError:
            print(f"Recording file {filename} not found. Creating empty playback.")
            self.playback_data = {'type': 'single', 'horizontal': [], 'vertical': [], 'target_pos': self.target_pos}

    def get_ai_input(self):
        """AI agent that controls either horizontal or vertical movement"""
        dx = self.target_pos[0] - self.cursor_pos[0]
        dy = self.target_pos[1] - self.cursor_pos[1]
        
        # AI strength (K). Adjust this to make AI 'weaker' or 'stronger'
        k = 0.035
        
        if self.ai_control_axis == AI_CONTROLS_VERTICAL:
            # AI controls up/down only
            return [0, dy * k]
        else:
            # AI controls left/right only
            return [dx * k, 0]

    def get_playback_input(self):
        """Get pre-recorded movement for current frame"""
        if self.playback_axis is None:
            print("Error: playback_axis not set before game start")
            return [0, 0]
        
        if self.playback_data.get('type') == 'session':
            iterations = self.playback_data.get('iterations', [])
            if 1 <= self.iteration <= len(iterations):
                current_axis = iterations[self.iteration - 1].get(f'partner_{self.playback_axis}', [])
            else:
                current_axis = []
        else:
            current_axis = self.playback_data.get(f'partner_{self.playback_axis}', self.playback_data.get(self.playback_axis, []))
        
        if self.playback_index < len(current_axis):
            velocity = current_axis[self.playback_index]
            self.playback_index += 1
            
            if self.playback_axis == 'horizontal':
                return [velocity, 0]
            else:
                return [0, velocity]
        
        return [0, 0]

    def log_frame(self):
        elapsed = time.time() - self.start_time
        # Recording: Time, CursorPos, HumanInput, PartnerInput, TargetPos
        self.data_log.append([
            elapsed, 
            self.cursor_pos[0], self.cursor_pos[1],
            self.human_v[0], self.human_v[1],
            self.partner_v[0], self.partner_v[1],
            self.target_pos[0], self.target_pos[1]
        ])

    def save_data(self):
        keys = ["timestamp", "cursor_x", "cursor_y", "h_vx", "h_vy", "p_vx", "p_vy", "target_x", "target_y"]
        if not os.path.exists(FILE_NAME):
            with open(FILE_NAME, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(keys)
        
        with open(FILE_NAME, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerows(self.data_log)
        print(f"Data saved to {FILE_NAME}")

    def draw_text(self, text, font, color, surface, x, y):
        """Draw text on the screen"""
        textobj = font.render(text, True, color)
        textrect = textobj.get_rect()
        textrect.topleft = (x, y)
        surface.blit(textobj, textrect)

    def run(self):
        """Run one iteration - ends when target is hit"""
        font = pygame.font.Font(None, 24)
        
        while self.running and not self.target_hit:
            self.screen.fill(WHITE)
            
            # 1. Event Handling
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    self.running = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        self.running = False

            # 2. Capture Human Input
            keys = pygame.key.get_pressed()
            self.human_v = [0, 0]
            speed = 5
            
            if self.mode == INDIVIDUAL:
                # Individual mode: player controls all axes with arrow keys
                if keys[pygame.K_LEFT]:  self.human_v[0] = -speed
                if keys[pygame.K_RIGHT]: self.human_v[0] = speed
                if keys[pygame.K_UP]:    self.human_v[1] = -speed
                if keys[pygame.K_DOWN]:  self.human_v[1] = speed
            
            elif self.mode == COOPERATIVE:
                # Cooperative: player controls left/right with arrow keys
                if keys[pygame.K_LEFT]:  self.human_v[0] = -speed
                if keys[pygame.K_RIGHT]: self.human_v[0] = speed
            
            elif self.mode == PLAYBACK:
                # Playback: player controls the opposite axis from playback
                if self.playback_axis == 'horizontal':
                    # Playback controls horizontal, so player controls vertical
                    if keys[pygame.K_UP]:   self.human_v[1] = -speed
                    if keys[pygame.K_DOWN]: self.human_v[1] = speed
                else:
                    # Playback controls vertical, so player controls horizontal
                    if keys[pygame.K_LEFT]:  self.human_v[0] = -speed
                    if keys[pygame.K_RIGHT]: self.human_v[0] = speed
            
            elif self.mode == AI:
                # AI mode: player controls whichever axis the AI doesn't control
                if self.ai_control_axis == AI_CONTROLS_VERTICAL:
                    # AI controls vertical, so player controls horizontal (arrow keys)
                    if keys[pygame.K_LEFT]:  self.human_v[0] = -speed
                    if keys[pygame.K_RIGHT]: self.human_v[0] = speed
                else:
                    # AI controls horizontal, so player controls vertical (W/S keys)
                    if keys[pygame.K_w]: self.human_v[1] = -speed
                    if keys[pygame.K_s]: self.human_v[1] = speed

            # 3. Capture Partner Input based on mode
            if self.mode == INDIVIDUAL:
                # No partner in individual mode
                self.partner_v = [0, 0]
            
            elif self.mode == COOPERATIVE:
                # Second player uses W/S for up/down only
                self.partner_v = [0, 0]
                if keys[pygame.K_w]: self.partner_v[1] = -speed
                if keys[pygame.K_s]: self.partner_v[1] = speed
            
            elif self.mode == PLAYBACK:
                # Get pre-recorded movements
                self.partner_v = self.get_playback_input()
            
            elif self.mode == AI:
                # AI agent controls one axis only (randomized per iteration)
                self.partner_v = self.get_ai_input()

            # 4. Joint Action: SUM THE VELOCITIES
            self.cursor_pos[0] += (self.human_v[0] + self.partner_v[0])
            self.cursor_pos[1] += (self.human_v[1] + self.partner_v[1])

            # Clamp cursor to screen bounds
            self.cursor_pos[0] = max(CURSOR_RADIUS, min(SCREEN_WIDTH - CURSOR_RADIUS, self.cursor_pos[0]))
            self.cursor_pos[1] = max(CURSOR_RADIUS, min(SCREEN_HEIGHT - CURSOR_RADIUS, self.cursor_pos[1]))

            # 5. Check Target Collision
            dist = ((self.cursor_pos[0]-self.target_pos[0])**2 + (self.cursor_pos[1]-self.target_pos[1])**2)**0.5
            if dist < TARGET_RADIUS:
                self.target_hit = True
                print(f"Target Hit! Iteration {self.iteration} complete.")

            # 6. Draw everything
            pygame.draw.circle(self.screen, RED, self.target_pos, TARGET_RADIUS) # Target
            pygame.draw.circle(self.screen, BLUE, (int(self.cursor_pos[0]), int(self.cursor_pos[1])), CURSOR_RADIUS) # Cursor
            
            # Draw mode indicator and iteration
            mode_text = f"Mode: {self.mode.upper()} | Iteration: {self.iteration}/{NUM_ITERATIONS}"
            self.draw_text(mode_text, font, BLACK, self.screen, 10, 10)
            
            # Draw controls info
            if self.mode == INDIVIDUAL:
                self.draw_text("Controls: Arrow Keys", font, BLACK, self.screen, 10, 35)
            elif self.mode == COOPERATIVE:
                self.draw_text("P1: LEFT/RIGHT arrows | P2: W/S keys", font, BLACK, self.screen, 10, 35)
            elif self.mode == PLAYBACK:
                if self.playback_axis == 'horizontal':
                    self.draw_text("Playback: LEFT/RIGHT (horizontal) | You: UP/DOWN (vertical)", font, BLACK, self.screen, 10, 35)
                else:
                    self.draw_text("Playback: UP/DOWN (vertical) | You: LEFT/RIGHT (horizontal)", font, BLACK, self.screen, 10, 35)
            elif self.mode == AI:
                ai_axis = "UP/DOWN" if self.ai_control_axis == AI_CONTROLS_VERTICAL else "LEFT/RIGHT"
                self.draw_text(f"AI controls: {ai_axis} | You control: {'LEFT/RIGHT' if self.ai_control_axis == AI_CONTROLS_VERTICAL else 'UP/DOWN'}", font, BLACK, self.screen, 10, 35)
            
            self.draw_text("Press ESC to quit", font, BLACK, self.screen, 10, 60)
            
            # 7. Log and Update
            self.log_frame()
            pygame.display.flip()
            self.clock.tick(FPS)

        if self.target_hit:
            self.save_data()

def save_session_recording(mode, session_records):
    """Save a full gameplay session as one JSON file with all iterations"""
    if not os.path.exists(RECORDING_DIR):
        os.makedirs(RECORDING_DIR)

    session_data = {
        'mode': mode,
        'num_iterations': len(session_records),
        'timestamp': int(time.time()),
        'iterations': session_records
    }

    filename = os.path.join(RECORDING_DIR, f"session_recording_{int(time.time())}.json")
    with open(filename, 'w') as f:
        json.dump(session_data, f, indent=2)

    print(f"Session recording saved to {filename}")
    return filename


def show_menu():
    """Display menu to select game mode"""
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("CogSci Joint Action Task - Mode Selection")
    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)
    
    selected = 0
    modes = [
        (INDIVIDUAL, "1. Individual Mode (You control alone)"),
        (COOPERATIVE, "2. Cooperative Mode (2 players, divide controls)"),
        (PLAYBACK, "3. Playback Mode (Play against recorded player)"),
        (AI, "4. AI Mode (Play against AI agent)")
    ]
    
    selecting_mode = True
    while selecting_mode:
        screen.fill(WHITE)
            
        # Title
        title = font.render("Select Game Mode", True, BLACK)
        screen.blit(title, (SCREEN_WIDTH // 2 - title.get_width() // 2, 50))
        # Information about iterations
        # Draw mode options
        for i, (mode_key, mode_text) in enumerate(modes):
            if i == selected:
                color = BLUE
                prefix = ">>> "
            else:
                color = BLACK
                prefix = "    "
            
            text = small_font.render(prefix + mode_text, True, color)
            screen.blit(text, (50, 150 + i * 60))
        
        # Information about iterations
        info_text = small_font.render(f"Each mode will run {NUM_ITERATIONS} times", True, GRAY)
        screen.blit(info_text, (SCREEN_WIDTH // 2 - info_text.get_width() // 2, 450))
        
        # Instructions
        instructions = small_font.render("Use UP/DOWN arrows to select, ENTER to confirm, ESC to exit", True, GRAY)
        screen.blit(instructions, (SCREEN_WIDTH // 2 - instructions.get_width() // 2, 500))
        
        pygame.display.flip()
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return None
            
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_UP:
                    selected = (selected - 1) % len(modes)
                elif event.key == pygame.K_DOWN:
                    selected = (selected + 1) % len(modes)
                elif event.key == pygame.K_RETURN:
                    selecting_mode = False
                elif event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return None
        
        clock.tick(FPS)
    
    pygame.quit()
    return modes[selected][0]


def save_session_recording(mode, session_records):
    """Save a full gameplay session as one JSON file with all iterations"""
    if not os.path.exists(RECORDING_DIR):
        os.makedirs(RECORDING_DIR)

    session_data = {
        'mode': mode,
        'num_iterations': len(session_records),
        'timestamp': int(time.time()),
        'iterations': session_records
    }

    filename = os.path.join(RECORDING_DIR, f"session_recording_{int(time.time())}.json")
    with open(filename, 'w') as f:
        json.dump(session_data, f, indent=2)

    print(f"Session recording saved to {filename}")
    return filename


def show_recording_menu():
    """Display menu to select a recording file for playback"""
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("CogSci Joint Action Task - Select Recording")
    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)
    
    # Get list of recordings
    recordings = []
    if os.path.exists(RECORDING_DIR):
        recordings = [f for f in os.listdir(RECORDING_DIR) if f.endswith('.json')]
        recordings.sort(reverse=True)
    
    if not recordings:
        screen.fill(WHITE)
        text = small_font.render("No recordings found. Running COOPERATIVE mode to record.", True, BLACK)
        screen.blit(text, (50, SCREEN_HEIGHT // 2))
        pygame.display.flip()
        pygame.time.wait(2000)
        pygame.quit()
        return None
    
    selected = 0
    selecting = True
    
    while selecting:
        screen.fill(WHITE)
        
        title = font.render("Select Recording to Play", True, BLACK)
        screen.blit(title, (SCREEN_WIDTH // 2 - title.get_width() // 2, 50))
        
        for i, recording in enumerate(recordings):
            if i == selected:
                color = BLUE
                prefix = ">>> "
            else:
                color = BLACK
                prefix = "    "
            
            text = small_font.render(prefix + recording, True, color)
            screen.blit(text, (50, 150 + i * 40))
        
        instructions = small_font.render("Use UP/DOWN to select, ENTER to confirm, ESC to cancel", True, GRAY)
        screen.blit(instructions, (SCREEN_WIDTH // 2 - instructions.get_width() // 2, 500))
        
        pygame.display.flip()
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return None
            
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_UP:
                    selected = (selected - 1) % len(recordings)
                elif event.key == pygame.K_DOWN:
                    selected = (selected + 1) % len(recordings)
                elif event.key == pygame.K_RETURN:
                    selecting = False
                elif event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return None
        
        clock.tick(FPS)
    
    pygame.quit()
    return os.path.join(RECORDING_DIR, recordings[selected])


def show_iteration_ready_screen(mode, iteration, total, ai_axis=None, playback_axis=None):
    """Display control instructions before each iteration - wait for SPACE to start"""
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("CogSci Joint Action Task - Ready for Next Trial")
    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)
    
    waiting = True
    cancelled = False
    
    while waiting:
        screen.fill(WHITE)
        
        # Iteration counter
        iteration_text = font.render(f"Iteration {iteration} of {total}", True, BLACK)
        screen.blit(iteration_text, (SCREEN_WIDTH // 2 - iteration_text.get_width() // 2, 50))
        
        # Display controls for this mode
        y_pos = 150
        controls_title = small_font.render("Controls:", True, BLACK)
        screen.blit(controls_title, (100, y_pos))
        y_pos += 40
        
        if mode == INDIVIDUAL:
            control_text = small_font.render("Use Arrow Keys to control the ball", True, BLACK)
            screen.blit(control_text, (120, y_pos))
        elif mode == COOPERATIVE:
            control_text1 = small_font.render("Player 1: LEFT/RIGHT Arrow Keys (horizontal)", True, BLACK)
            control_text2 = small_font.render("Player 2: W/S Keys (vertical)", True, BLACK)
            screen.blit(control_text1, (120, y_pos))
            screen.blit(control_text2, (120, y_pos + 30))
        elif mode == PLAYBACK:
            if playback_axis == 'horizontal':
                control_text1 = small_font.render("Playback player: LEFT/RIGHT (horizontal)", True, BLACK)
                control_text2 = small_font.render("You control: UP/DOWN (vertical)", True, BLACK)
            else:
                control_text1 = small_font.render("Playback player: UP/DOWN (vertical)", True, BLACK)
                control_text2 = small_font.render("You control: LEFT/RIGHT (horizontal)", True, BLACK)
            screen.blit(control_text1, (120, y_pos))
            screen.blit(control_text2, (120, y_pos + 30))
        elif mode == AI:
            ai_control = "UP/DOWN (W/S)" if ai_axis == AI_CONTROLS_VERTICAL else "LEFT/RIGHT (Arrows)"
            player_control = "LEFT/RIGHT (Arrows)" if ai_axis == AI_CONTROLS_VERTICAL else "UP/DOWN (W/S)"
            control_text1 = small_font.render(f"AI controls: {ai_control}", True, BLACK)
            control_text2 = small_font.render(f"You control: {player_control}", True, BLACK)
            screen.blit(control_text1, (120, y_pos))
            screen.blit(control_text2, (120, y_pos + 30))
        
        # Goal
        goal_y = 320
        goal_text = small_font.render("Goal: Direct the BLUE ball into the RED target", True, BLACK)
        screen.blit(goal_text, (100, goal_y))
        
        # Ready text
        ready_text = font.render("Press SPACE to start", True, BLUE)
        screen.blit(ready_text, (SCREEN_WIDTH // 2 - ready_text.get_width() // 2, 420))
        
        # ESC to quit hint
        esc_text = small_font.render("Press ESC to quit and return to menu", True, GRAY)
        screen.blit(esc_text, (SCREEN_WIDTH // 2 - esc_text.get_width() // 2, 500))
        
        pygame.display.flip()
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return False
            
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:
                    waiting = False
                elif event.key == pygame.K_ESCAPE:
                    cancelled = True
                    waiting = False
        
        clock.tick(FPS)
    
    pygame.quit()
    return not cancelled


if __name__ == "__main__":
    while True:
        # Show mode selection menu
        mode = show_menu()

        if mode is None:
            print("Exiting game.")
            break

        recording_file = None

        # If playback mode, let user select a recording
        if mode == PLAYBACK:
            recording_file = show_recording_menu()
            if recording_file is None:
                print("No recording selected. Returning to menu.")
                continue

        # Create shuffled order of target indices for this game mode
        target_order = list(range(NUM_ITERATIONS))
        random.shuffle(target_order)

        # Run NUM_ITERATIONS of the selected mode
        session_records = []
        user_cancelled = False
        for iteration in range(1, NUM_ITERATIONS + 1):
            # Generate AI axis if AI mode (randomized for each iteration)
            ai_axis = None
            if mode == AI:
                ai_axis = random.randint(0, 1)
            
            # Pre-determine playback axis for PLAYBACK mode
            playback_axis = None
            if mode == PLAYBACK:
                playback_axis = random.choice(['horizontal', 'vertical'])
                print(f"Iteration {iteration}: Playback will use {playback_axis} axis")
            
            # Get the shuffled target for this iteration
            target_pos = PRESET_TARGETS[target_order[iteration - 1]]
            
            # Show ready screen with controls
            if not show_iteration_ready_screen(mode, iteration, NUM_ITERATIONS, ai_axis=ai_axis, playback_axis=playback_axis):
                print(f"User cancelled during iteration {iteration}. Returning to menu.")
                user_cancelled = True
                break
            
            # Create and run the game for this iteration
            game = ExperimentGame(mode=mode, recording_file=recording_file, iteration=iteration, ai_axis=ai_axis, target_pos=target_pos, playback_axis=playback_axis)
            game.run()
            
            # Check if user pressed ESC during game
            if not game.running:
                print(f"User cancelled during iteration {iteration}. Returning to menu.")
                user_cancelled = True
                break

            if mode == COOPERATIVE:
                human_horizontal = [entry[3] for entry in game.data_log]
                human_vertical = [entry[4] for entry in game.data_log]
                partner_horizontal = [entry[5] for entry in game.data_log]
                partner_vertical = [entry[6] for entry in game.data_log]
                session_records.append({
                    'iteration': iteration,
                    'target_pos': game.target_pos,
                    'human_horizontal': human_horizontal,
                    'human_vertical': human_vertical,
                    'partner_horizontal': partner_horizontal,
                    'partner_vertical': partner_vertical,
                    'horizontal': partner_horizontal,
                    'vertical': partner_vertical
                })

        if not user_cancelled:
            if mode == COOPERATIVE and session_records:
                recording_file = save_session_recording(mode, session_records)
                print(f"\nSession recording saved: {recording_file}")
            print(f"\nCompleted all {NUM_ITERATIONS} iterations of {mode.upper()} mode!")
            print("Returning to menu...")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
Target Hit! Iteration 1 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 2 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 3 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 4 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 5 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 6 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 7 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 8 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 9 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 10 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 11 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 12 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 13 complete.
Data saved to experiment_data.csv